# SpliceBERT Head–Context Atlas — discovery screen
Runs all 96 heads and six MLPs on the balanced **discovery-only** atlas. The order is enforced: verify → tests → cross-device smoke → benchmark → masked-health panel → explicit GO → target screen → validate → merge. No validation or sealed chromosomes are bundled.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, shutil, subprocess, json, hashlib
ZIP = pathlib.Path('/content/drive/MyDrive/gfm_atlas_screen_colab_v1.zip')
UNPACK = pathlib.Path('/content/gfm_atlas_screen')
if UNPACK.exists(): shutil.rmtree(UNPACK)
subprocess.run(['unzip', '-q', str(ZIP), '-d', '/content'], check=True)
ROOT = pathlib.Path('/content/gfm_atlas_screen/repo')
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/gfm_atlas_screen_v1_results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
ENV = {**os.environ, 'PYTHONHASHSEED':'0', 'TOKENIZERS_PARALLELISM':'false'}
print(ROOT, DRIVE_OUT)

In [ ]:
# Keep Colab's CUDA-enabled torch. Never pip-install torch here.
subprocess.run(['pip','install','-q','-r','studies/splicebert_head_context_atlas/colab_atlas_package/requirements-colab.txt'], check=True)

In [ ]:
# Verify every bundled byte before loading the model.
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
bundle=json.loads((ROOT/'BUNDLE_MANIFEST.json').read_text())
bad=[]
for relative, rec in bundle['files'].items():
    path=ROOT/relative
    if not path.is_file() or path.stat().st_size!=rec['size_bytes'] or sha256(path)!=rec['sha256']: bad.append(relative)
assert not bad, f'bundle verification failed: {bad[:5]}'
import torch
assert torch.cuda.is_available(), 'GPU runtime required: Runtime → Change runtime type → GPU'
print('Bundle OK:',len(bundle['files']),'files | GPU:',torch.cuda.get_device_name(0),'| torch',torch.__version__)

In [ ]:
# Mechanical/unit suite, including both coordinate contracts and real-model vector/scalar agreement.
subprocess.run(['pytest','-q','tests/test_atlas_context.py','tests/test_atlas_matching.py','tests/test_competition.py','tests/test_head_context_screen.py'], check=True, env=ENV)

In [ ]:
# Decisive cross-device smoke: 2,448 effects must reproduce the local preflight within 0.002.
SMOKE=pathlib.Path('/content/atlas_screen_smoke')
if SMOKE.exists(): shutil.rmtree(SMOKE)
subprocess.run(['python','scripts/splicebert_head_context_atlas/run_atlas_discovery_screen.py','--phase','both','--batch','4','--packet-ids-from-summary','results/splicebert_head_context_atlas/atlas_model_preflight_v1.summary.json','--reference-rows','results/splicebert_head_context_atlas/atlas_model_preflight_v1.rows.jsonl','--out',str(SMOKE)],check=True,env=ENV)
subprocess.run(['python','scripts/splicebert_head_context_atlas/validate_atlas_discovery_screen.py','--out',str(SMOKE),'--manifest-mode','both','--packet-ids-from-summary','results/splicebert_head_context_atlas/atlas_model_preflight_v1.summary.json'],check=True,env=ENV)
smoke=json.loads((SMOKE/'run_manifest.both.json').read_text())
print(json.dumps(smoke['reference_verification'],indent=2))

In [ ]:
# 32-packet target benchmark. Review projected GPU hours/storage and peak memory.
BENCH=pathlib.Path('/content/atlas_screen_benchmark')
if BENCH.exists(): shutil.rmtree(BENCH)
BATCH=16
subprocess.run(['python','scripts/splicebert_head_context_atlas/run_atlas_discovery_screen.py','--phase','target','--n','32','--batch',str(BATCH),'--out',str(BENCH)],check=True,env=ENV)
bench=json.loads((BENCH/'run_manifest.target.json').read_text())
print(json.dumps(bench['phase_reports']['target'],indent=2))

In [ ]:
# Low-cost health phase FIRST. Resumable in Drive; must pass before target inference.
subprocess.run(['python','scripts/splicebert_head_context_atlas/run_atlas_discovery_screen.py','--phase','health','--health-n','512','--batch',str(BATCH),'--out',str(DRIVE_OUT)],check=True,env=ENV)
health=json.loads((DRIVE_OUT/'run_manifest.health.json').read_text())
assert health['status']=='COMPLETE' and health['health']['gate'], health['failures']
print(json.dumps({'phase':health['phase_reports']['health'],'health':health['health']},indent=2))

## Explicit target-run gate
Confirm: smoke passed, benchmark is within budget, health passed, output is on Drive. Then set `LAUNCH_TARGET=True`. The target run resumes by skipping completed packet shards.

In [ ]:
LAUNCH_TARGET=False
assert LAUNCH_TARGET, 'Review benchmark + health, then explicitly set LAUNCH_TARGET=True'
subprocess.run(['python','scripts/splicebert_head_context_atlas/run_atlas_discovery_screen.py','--phase','target','--batch',str(BATCH),'--out',str(DRIVE_OUT)],check=True,env=ENV)

In [ ]:
# Validate every shard/hash/component grid, then merge only if audit PASS.
subprocess.run(['python','scripts/splicebert_head_context_atlas/validate_atlas_discovery_screen.py','--out',str(DRIVE_OUT),'--manifest-mode','separate'],check=True,env=ENV)
MERGED=DRIVE_OUT/'atlas_discovery_v1'
subprocess.run(['python','scripts/splicebert_head_context_atlas/merge_atlas_discovery_screen.py','--screen',str(DRIVE_OUT),'--prefix',str(MERGED)],check=True,env=ENV)
print(json.loads((pathlib.Path(str(MERGED)+'.summary.json')).read_text()))